### Mugrade boilerplate

In [ ]:
### Run this cell to install and import the homework tests
!pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part4_inference_tests.py

import mugrade
import os
from part4_inference_tests import *
os.environ["MUGRADE_HW"] = "Part 4 - Inference"
os.environ["MUGRADE_KEY"] = "" ### Your key here


### BPE

Paste your BPE code from Part 1 here.

In [ ]:
### BEGIN YOUR CODE
pass
### END YOUR CODE


### LLM with efficient inference

Add KV caching and position-based RoPE to the LLM architecture from Part 2, then implement `generate()` to produce completions.

The new graded methods are `rope`, `multihead_attn`, `transformer_block`, `__call__`, and `generate`. Reuse the other functions from Part 2; they are not graded again here.

`generate` takes a batch of equally long prompts and returns a tensor containing each prompt followed by at most `max_tokens` new tokens. Sample from the last-token logits using the positive temperature `temp`. Pass an EOS token and stop each sequence when it generates that token. Padding after a sequence's first generated EOS is not graded. A zero token budget returns the prompts.


In [ ]:
import torch
import math

def embedding(tokens, weight, dtype):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

def linear(x, weight):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

def silu(x):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

def rms_norm(x):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

def softmax(x):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

def self_attn(q,k,v,mask):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE


class LLM:
    # @mugrade.local_tests
    def rope(self, x, pos=0):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def multihead_attn(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def mlp(self, x, layer):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def transformer_block(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def __init__(self, config):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def __call__(self, tokens, pos=0, cache = None):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE
    
    # @mugrade.local_tests
    def generate(self, prompts, max_tokens=500, temp=0.7, eos=None):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def save(self, filename):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    def load(self, filename):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE


### Generating text

After you have implmeneted the functions above, you can use the following code to download the d30 model we demonstrated in class (you're also welcome to use your own d12 model if you've been able to train one, but the d30 uses the exact same architecture).  You should be able to use your new architecture to generate samples from the model.

In [ ]:
from huggingface_hub import hf_hub_download
import os
import json

repo = "zkolter/llm_speedrun"
filenames = ["llm.d30.pt", "config.d30.json", "tokenizer_50M.bpe"]

for filename in filenames:
    if not os.path.exists(filename):
        hf_hub_download(repo_id=repo, filename=filename, repo_type="dataset", local_dir=".")

In [ ]:
with open("config.d30.json", "rt") as f:
    config = json.load(f)
torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}
config["dtype"] = torch_types[config["dtype"]]

llm = LLM(config)
llm.load("llm.d30.pt")
for k in llm.params: llm.params[k] = llm.params[k].cuda()
for k in llm.buffers: llm.buffers[k] = llm.buffers[k].cuda()
bpe = BPE(config["tokenizer"])

In [ ]:
prompt = "<DOCUMENT><USER>WHy is the sky blue?</USER><ASSISTANT>"
prompts = torch.tensor([bpe.encode(prompt)]).cuda()

eos = bpe.vocab.index("</ASSISTANT>")
tokens = llm.generate(prompts, max_tokens=500, temp=0.7, eos=eos)
print(bpe.decode(tokens.tolist()[0]))